In [2]:
"""
==================================================================
E-COMMERCE DATA CLEANING — Phase 1
Simple flow: IMPORT -> READ RAW CSV -> CLEAN -> SAVE CLEAN CSV
The same pattern is followed for every table.

Folder structure expected:
data/raw/.csv         <- input (messy data)
data/processed/.csv   <- output (clean data) - this will be created

Run:     python 01_data_cleaning.py
===================================================================
"""

'\n==================================================================\nE-COMMERCE DATA CLEANING — Phase 1\nSimple flow: IMPORT -> READ RAW CSV -> CLEAN -> SAVE CLEAN CSV\nThe same pattern is followed for every table.\n\nFolder structure expected:\ndata/raw/.csv         <- input (messy data)\ndata/processed/.csv   <- output (clean data) - this will be created\n\nRun:     python 01_data_cleaning.py\n===================================================================\n'

In [3]:
# ------------------------------------------------------------------
# STEP 1: IMPORT LIBRARIES
# ------------------------------------------------------------------
import pandas as pd
import numpy as np


In [4]:
import os

# ------------------------------------------------------------------
# STEP 2: FOLDER PATHS (you can change this according to your system)
# ------------------------------------------------------------------
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"

# Ensure output folder exists so saving files won't throw an error
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [9]:
# ==================================================================
# TABLE 1: CUSTOMERS
# ==================================================================
print("Cleaning customers...")

# --- IMPORT (read raw csv) ---
customers = pd.read_csv(f"{RAW_DIR}/customers.csv")
print("Raw rows:", len(customers))

# --- CLEAN ---

# 1. duplicate rows hatao
customers = customers.drop_duplicates()

# 2. duplicate customer_id hatao (agar same customer_id do baar hai to pehla rakho)
customers = customers.drop_duplicates(subset="customer_id", keep="first")

# 3. text columns ka extra space aur case fix karo
text_cols = ["country", "city", "device", "acquisition_channel", "customer_segment"]
for col in text_cols:
    customers[col] = customers[col].astype(str).str.strip()          # extra space hatao
    customers[col] = customers[col].replace({"nan": np.nan, "": np.nan})  # "nan" string ko real NaN banao

customers["country"] = customers["country"].str.title()   # "india" -> "India"
customers["city"] = customers["city"].str.title()

# 4. country naming fix karo (Usa -> USA, Uk -> UK)
customers["country"] = customers["country"].replace({"Usa": "USA", "Uk": "UK", "Uae": "UAE"})

# 5. missing values fill karo
customers["city"] = customers["city"].fillna("Unknown")
customers["device"] = customers["device"].fillna("Unknown")
customers["customer_segment"] = customers["customer_segment"].fillna("Unclassified")

# 6. date column ko proper date format mein convert karo
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
customers = customers.dropna(subset=["signup_date"])   # jo date convert nahi hui, wo row hatao

print("Clean rows:", len(customers))

# --- SAVE ---
customers.to_csv(f"{PROCESSED_DIR}/cl_customers.csv", index=False)
print("Saved: cl_customers.csv\n")


Cleaning customers...
Raw rows: 15075
Clean rows: 15000


Saved: cl_customers.csv



In [10]:
# ==================================================================
# TABLE 2: PRODUCTS
# ==================================================================
print("Cleaning products...")

products = pd.read_csv(f"{RAW_DIR}/products.csv")
print("Raw rows:", len(products))

# Remove duplicate product_ids
products = products.drop_duplicates(subset="product_id", keep="first")

# Clean text columns and handle "Nan" strings properly
text_cols = ["category", "subcategory", "brand"]
for col in text_cols:
    products[col] = products[col].replace({"nan": np.nan, "": np.nan})  # Convert string NaNs back to real NaN
    products[col] = products[col].astype(str).str.strip().str.title()
    products[col] = products[col].replace({"Nan": np.nan})

# Flag missing cost before filling
products["cost_is_estimated"] = products["cost"].isna()

# Fill missing cost with category median (fallback to overall median if category median is missing)
category_median = products.groupby("category")["cost"].transform("median")
overall_median = products["cost"].median()
products["cost"] = products["cost"].fillna(category_median).fillna(overall_median)

# Flag invalid selling prices (0 or negative)
products["price_is_invalid"] = products["selling_price"] <= 0

print("Clean rows:", len(products))

# Save cleaned products data
products.to_csv(f"{PROCESSED_DIR}/cl_products.csv", index=False)
print("Saved: cl_products.csv\n")

Cleaning products...
Raw rows: 798
Clean rows: 798
Saved: cl_products.csv



In [11]:
# ==================================================================
# TABLE 3: ORDERS
# ==================================================================
print("Cleaning orders...")

orders = pd.read_csv(f"{RAW_DIR}/orders.csv")
print("Raw rows:", len(orders))

# Remove duplicate order_id
orders = orders.drop_duplicates(subset="order_id", keep="first")

# Fix date format and drop invalid dates
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders = orders.dropna(subset=["order_date"])

# Remove orphan orders (orders whose customer_id does not exist in the customers table)
valid_customers = set(customers["customer_id"])
orders = orders[orders["customer_id"].isin(valid_customers)]

# Clean payment_method text format
orders["payment_method"] = orders["payment_method"].astype(str).str.strip().str.title()
orders["payment_method"] = orders["payment_method"].replace({"Nan": np.nan})

# Fill missing discount/shipping cost with 0
orders["discount"] = orders["discount"].fillna(0)
orders["shipping_cost"] = orders["shipping_cost"].fillna(0)

# Keep only valid order statuses
valid_status = ["Completed", "Cancelled", "Returned", "Pending"]
orders = orders[orders["order_status"].isin(valid_status)]

print("Clean rows:", len(orders))

orders.to_csv(f"{PROCESSED_DIR}/cl_orders.csv", index=False)
print("Saved: cl_orders.csv\n")

Cleaning orders...


Raw rows: 60180


Clean rows: 59985


Saved: cl_orders.csv



In [12]:
# ==================================================================
# TABLE 4: ORDER_ITEMS
# ==================================================================
print("Cleaning order_items...")

order_items = pd.read_csv(f"{RAW_DIR}/order_items.csv")
print("Raw rows:", len(order_items))

# Remove duplicate order_item_ids
order_items = order_items.drop_duplicates(subset="order_item_id", keep="first")

# Remove orphan records (order_id or product_id not present in cleaned orders or products tables)
valid_orders = set(orders["order_id"])
valid_products = set(products["product_id"])
order_items = order_items[order_items["order_id"].isin(valid_orders)]
order_items = order_items[order_items["product_id"].isin(valid_products)]

# Remove invalid quantities (0 or negative values)
order_items = order_items[order_items["quantity"] > 0]

# Calculate line_total (useful for downstream revenue calculations)
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

print("Clean rows:", len(order_items))

# Save cleaned order_items data
order_items.to_csv(f"{PROCESSED_DIR}/cl_order_items.csv", index=False)
print("Saved: cl_order_items.csv\n")

Cleaning order_items...


Raw rows: 119134


Clean rows: 119091


Saved: cl_order_items.csv



In [13]:
# ==================================================================
# TABLE 5: WEB_EVENTS
# ==================================================================
print("Cleaning web_events...")

web_events = pd.read_csv(f"{RAW_DIR}/web_events.csv")
print("Raw rows:", len(web_events))

# Remove duplicate event_ids
web_events = web_events.drop_duplicates(subset="event_id", keep="first")

# Fix timestamp format and drop invalid timestamps
web_events["event_timestamp"] = pd.to_datetime(web_events["event_timestamp"], errors="coerce")
web_events = web_events.dropna(subset=["event_timestamp"])

# Remove orphan events (customer_ids not present in the cleaned customers table)
web_events = web_events[web_events["customer_id"].isin(valid_customers)]

# Fill missing session IDs with a placeholder
web_events["session_id"] = web_events["session_id"].fillna("UNKNOWN_SESSION")

# Keep only valid event types
valid_events = ["page_view", "product_view", "add_to_cart", "checkout", "purchase", "login", "wishlist"]
web_events = web_events[web_events["event_name"].isin(valid_events)]

print("Clean rows:", len(web_events))

# Save cleaned web_events data
web_events.to_csv(f"{PROCESSED_DIR}/cl_web_events.csv", index=False)
print("Saved: cl_web_events.csv\n")

Cleaning web_events...


Raw rows: 292069


Clean rows: 292069


Saved: cl_web_events.csv



In [15]:
# ==================================================================
# TABLE 6: MARKETING_CAMPAIGNS
# ==================================================================
print("Cleaning marketing_campaigns...")

marketing = pd.read_csv(f"{RAW_DIR}/marketing_campaigns.csv")
print("Raw rows:", len(marketing))

# Remove duplicate records
marketing = marketing.drop_duplicates()

# Fix date format and drop invalid dates
marketing["date"] = pd.to_datetime(marketing["date"], errors="coerce")
marketing = marketing.dropna(subset=["date"])

# Fill missing spend with 0
marketing["spend"] = marketing["spend"].fillna(0)

# Fix logical errors: clicks cannot exceed impressions, and conversions cannot exceed clicks
marketing = marketing[marketing["clicks"] <= marketing["impressions"]]
marketing = marketing[marketing["conversions"] <= marketing["clicks"]]

print("Clean rows:", len(marketing))

# Save cleaned marketing_campaigns data
marketing.to_csv(f"{PROCESSED_DIR}/cl_marketing_campaigns.csv", index=False)
print("Saved: cl_marketing_campaigns.csv\n")

Cleaning marketing_campaigns...
Raw rows: 13834
Clean rows: 13834


Saved: cl_marketing_campaigns.csv



In [16]:
# ==================================================================
# TABLE 7: PAYMENTS
# ==================================================================
print("Cleaning payments...")

payments = pd.read_csv(f"{RAW_DIR}/payments.csv")
print("Raw rows:", len(payments))

# Remove duplicate payment_ids
payments = payments.drop_duplicates(subset="payment_id", keep="first")

# Remove orphan payments (order_id not present in cleaned orders table)
payments = payments[payments["order_id"].isin(valid_orders)]

# Fix payment date format and drop invalid dates
payments["payment_date"] = pd.to_datetime(payments["payment_date"], errors="coerce")
payments = payments.dropna(subset=["payment_date"])

# Remove invalid amounts (negative values)
payments = payments[payments["amount"] >= 0]

print("Clean rows:", len(payments))

# Save cleaned payments data
payments.to_csv(f"{PROCESSED_DIR}/cl_payments.csv", index=False)
print("Saved: cl_payments.csv\n")

Cleaning payments...
Raw rows: 34368


Clean rows: 34357


Saved: cl_payments.csv



In [17]:
# ==================================================================
# TABLE 8: RETURNS
# ==================================================================
print("Cleaning returns...")

returns = pd.read_csv(f"{RAW_DIR}/returns.csv")
print("Raw rows:", len(returns))

# Remove duplicate return_ids
returns = returns.drop_duplicates(subset="return_id", keep="first")

# Remove orphan returns (order_id not present in cleaned orders table)
returns = returns[returns["order_id"].isin(valid_orders)]

# Fix return date format and drop invalid dates
returns["return_date"] = pd.to_datetime(returns["return_date"], errors="coerce")
returns = returns.dropna(subset=["return_date"])

# Fill missing return reason with default value
returns["return_reason"] = returns["return_reason"].fillna("Not Specified")

print("Clean rows:", len(returns))

# Save cleaned returns data
returns.to_csv(f"{PROCESSED_DIR}/cl_returns.csv", index=False)
print("Saved: cl_returns.csv\n")

Cleaning returns...
Raw rows: 5454


Clean rows: 5454
Saved: cl_returns.csv



In [19]:
# ==================================================================
# TABLE 9: EXPERIMENTS
# ==================================================================
print("Cleaning experiments...")

experiments = pd.read_csv(f"{RAW_DIR}/experiments.csv")
print("Raw rows:", len(experiments))

# Remove duplicate records
experiments = experiments.drop_duplicates()

# Fix experiment date format and drop invalid dates
experiments["experiment_date"] = pd.to_datetime(experiments["experiment_date"], errors="coerce")
experiments = experiments.dropna(subset=["experiment_date"])

# Keep only valid variants
experiments = experiments[experiments["variant"].isin(["control", "treatment"])]

print("Clean rows:", len(experiments))

# Save cleaned experiments data
experiments.to_csv(f"{PROCESSED_DIR}/cl_experiments.csv", index=False)
print("Saved: cl_experiments.csv\n")


# ==================================================================
# DONE
# ==================================================================
print("=" * 50)
print("ALL TABLES CLEANED AND SAVED TO:", PROCESSED_DIR)
print("=" * 50)

Cleaning experiments...
Raw rows: 8000
Clean rows: 8000
Saved: cl_experiments.csv

ALL TABLES CLEANED AND SAVED TO: data/processed


In [18]:
'cleaning ok'

'cleaning ok'